In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [16]:
# Load the dataset
df = pd.read_csv("dados_manutencao.csv")

# Display the first 5 rows
print(df.head().to_markdown(index=False, numalign="left", stralign="left"))

# Print the column names and their data types
print(df.info())

| Data de Produção Acumulada   | Cod. Ordem   | Cod Recurso   | Cod Produto   | Qt. Total Acumulada Produzida até a data específica   | Qt. Acumulada Refugada até a data específica   | Qtd. Acumulada total Retrabalhada até a data específica   | Fator Un.   | Cód. Un.   | Descrição da massa (Composto)   | Consumo total de Massa Acumulada   | Tempo Restante para Manutenção   |
|:-----------------------------|:-------------|:--------------|:--------------|:------------------------------------------------------|:-----------------------------------------------|:----------------------------------------------------------|:------------|:-----------|:--------------------------------|:-----------------------------------|:---------------------------------|
| 2023-09-08                   | 4458603      | IJ-117        | SA02961       | 87                                                    | 98                                             | 14                                                        |

* As colunas Data de Produção Acumulada, Cod. Ordem, Cod Recurso, Cod Produto, Fator Un., Cód. Un., Descrição da massa (Composto), and Tempo Restante para Manutenção não são relevantes para nossa análise, então vou removê-las. As colunas Qt. Total Acumulada Produzida até a data específica, Qt. Acumulada Refugada até a data específica, and Qtd. Acumulada total Retrabalhada até a data específica representam quantidades acumuladas até uma data específica, então vou renomeá-las para Qtd_Produzida, Qtd_Refugada e Qtd_Retrabalhada, respectivamente. A coluna Consumo total de Massa Acumulada representa o consumo total de massa, então vou renomeá-la para Consumo_Massa_Total.

In [17]:
# Rename columns
df = df.rename(
    columns={
        "Qt. Total Acumulada Produzida até a data específica": "Qtd_Produzida",
        "Qt. Acumulada Refugada até a data específica": "Qtd_Refugada",
        "Qtd. Acumulada total Retrabalhada até a data específica": "Qtd_Retrabalhada",
        "Consumo total de Massa Acumulada": "Consumo_Massa_Total",
    }
)

# Drop unnecessary columns
df = df.drop(
    [
        "Data de Produção Acumulada",
        "Cod. Ordem",
        "Cod Recurso",
        "Cod Produto",
        "Fator Un.",
        "Cód. Un.",
        "Descrição da massa (Composto)",
        "Tempo Restante para Manutenção",
    ],
    axis=1,
)

# Display the first 5 rows
print(df.head().to_markdown(index=False, numalign="left", stralign="left"))

# Print the column names and their data types
print(df.info())

| Qtd_Produzida   | Qtd_Refugada   | Qtd_Retrabalhada   | Consumo_Massa_Total   |
|:----------------|:---------------|:-------------------|:----------------------|
| 87              | 98             | 14                 | 0.909355              |
| 819             | 1              | 8                  | 0.796544              |
| 84              | 74             | 4                  | 0.686085              |
| 868             | 19             | 47                 | 1.22212               |
| 95              | 69             | 25                 | 0.753044              |
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Qtd_Produzida        1000 non-null   int64  
 1   Qtd_Refugada         1000 non-null   int64  
 2   Qtd_Retrabalhada     1000 non-null   int64  
 3   Consumo_Massa_Total  1000 non-null   float64
dtypes: float64(1), i

In [18]:
# Gráfico de dispersão para visualizar a relação entre as variáveis e o consumo total de massa.
import altair as alt

# Create a scatter plot for each column
scatter_produzida = (
    alt.Chart(df)
    .mark_point()
    .encode(
        x=alt.X("Qtd_Produzida", title="Qtd_Produzida"),
        y=alt.Y("Consumo_Massa_Total", title="Consumo_Massa_Total"),
        tooltip=["Qtd_Produzida", "Consumo_Massa_Total"],
    )
    .properties(title="Qtd_Produzida vs Consumo_Massa_Total")
    .interactive()
)

scatter_refugada = (
    alt.Chart(df)
    .mark_point()
    .encode(
        x=alt.X("Qtd_Refugada", title="Qtd_Refugada"),
        y=alt.Y("Consumo_Massa_Total", title="Consumo_Massa_Total"),
        tooltip=["Qtd_Refugada", "Consumo_Massa_Total"],
    )
    .properties(title="Qtd_Refugada vs Consumo_Massa_Total")
    .interactive()
)

scatter_retrabalhada = (
    alt.Chart(df)
    .mark_point()
    .encode(
        x=alt.X("Qtd_Retrabalhada", title="Qtd_Retrabalhada"),
        y=alt.Y("Consumo_Massa_Total", title="Consumo_Massa_Total"),
        tooltip=["Qtd_Retrabalhada", "Consumo_Massa_Total"],
    )
    .properties(title="Qtd_Retrabalhada vs Consumo_Massa_Total")
    .interactive()
)

In [24]:
# Combine the plots into a single figure
chart = scatter_produzida | scatter_refugada | scatter_retrabalhada
path_class = "classification_images/"
# Save the chart
chart.save(path_class + "scatter_plots.json")
chart.save(path_class + "scatter_plots.png")

# Previsão de Consumo de Massa Total

Vou usar as colunas `Qtd_Produzida`, `Qtd_Refugada` e `Qtd_Retrabalhada` para prever a coluna `Consumo_Massa_Total` usando os seguintes algoritmos de regressão:

* Regressão Linear
* Regressão Ridge
* Regressão Lasso
* Decision Tree Regressor
* Random Forest Regressor
* Gradient Boosting Regressor

Vou avaliar o desempenho de cada modelo usando as métricas:

* **MSE (Mean Squared Error)**
* **R-squared**

In [26]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [27]:
# Split into features and target
X = df.drop("Consumo_Massa_Total", axis=1)
y = df["Consumo_Massa_Total"]

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train and evaluate models
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=1.0),
    "DecisionTreeRegressor": DecisionTreeRegressor(),
    "RandomForestRegressor": RandomForestRegressor(n_estimators=100),
    "GradientBoostingRegressor": GradientBoostingRegressor(),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"{name}:")
    print(f"  Mean Squared Error: {mse:.2f}")
    print(f"  R-squared: {r2:.2f}")

LinearRegression:
  Mean Squared Error: 0.09
  R-squared: 0.00
Ridge:
  Mean Squared Error: 0.09
  R-squared: 0.00
Lasso:
  Mean Squared Error: 0.09
  R-squared: -0.00
DecisionTreeRegressor:
  Mean Squared Error: 0.16
  R-squared: -0.84
RandomForestRegressor:
  Mean Squared Error: 0.10
  R-squared: -0.17
GradientBoostingRegressor:
  Mean Squared Error: 0.10
  R-squared: -0.10


## Análise dos Resultados dos Modelos de Regressão

Com base nos resultados, podemos observar que os modelos lineares (LinearRegression, Ridge e Lasso) apresentam o menor erro quadrático médio (MSE), mas um R-quadrado próximo de zero indica que esses modelos não conseguem explicar a variabilidade nos dados. O novo esquema com dados reais deve substituir essa tabela gerada com dados aleatórios para análise.

Os modelos baseados em árvore (DecisionTreeRegressor, RandomForestRegressor e GradientBoostingRegressor) têm um MSE maior e um R-quadrado negativo, sugerindo que eles têm um desempenho pior do que os modelos lineares para este conjunto de dados aleatórios.

### Observações:

* O conjunto de dados pode não ter uma relação linear forte entre as variáveis preditoras e a variável alvo, o que pode explicar o baixo desempenho dos modelos lineares.
* Os modelos baseados em árvore podem estar sofrendo de overfitting, levando a um desempenho ruim nos dados de teste.
* Ajustar os hiperparâmetros dos modelos ou usar algoritmos de regressão mais complexos pode melhorar o desempenho do modelo.